In [1]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Statsmodels
import statsmodels.api as sm


import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             confusion_matrix, classification_report,
                             brier_score_loss)


In [2]:

# EDA from week2 Mod B
df_diabetes = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")
df_ckd = pd.read_csv("Chronic_Kidney_Dsease_data.csv")
df_hypertension = pd.read_csv("hypertension_dataset.csv")
df_alzheimers = pd.read_csv("alzheimers_disease_data.csv")

# Generate basic summaries
diabetes_desc = df_diabetes.describe(include='all')
ckd_desc = df_ckd.describe(include='all')
hypertension_desc = df_hypertension.describe(include='all')
alzheimers_desc = df_alzheimers.describe(include='all')

# Check for duplicates
diabetes_duplicates = df_diabetes.duplicated().sum()
ckd_duplicates = df_ckd.duplicated().sum()
alzheimers_duplicates = df_alzheimers.duplicated().sum()
_duplicates = df_hypertension.duplicated().sum()
# Check for nulls
diabetes_nulls = df_diabetes.isnull().sum()
ckd_nulls = df_ckd.isnull().sum()
hypertension_nulls = df_hypertension.isnull().sum()
alzheimers__nulls = df_alzheimers.isnull().sum()

# Display descriptive summaries

print("\n===alzheimers-Summary ===")
print(ckd_desc)
# Return duplicates and total null values per dataset
print("\n=== Duplicates in alzheimers_duplicates Dataset ===")
print(f"alzheimers: {alzheimers_duplicates}")

# Return nulls
print("\n=== Total Null Values in alzheimers Dataset ===")
print(f"Diabealzheimerstes: {alzheimers__nulls.sum()}")


#Diabetes Dataset
#Duplicates: 24,206 rows — significant duplication, should be reviewed or removed
#Missing values: None
#Usability: Usable after deduplication

#Chronic Kidney Disease (CKD) Dataset
#Duplicates: None
#Missing values: None\
#Usability: Clean and ready for analysis

#Hypertension Dataset
#Duplicates: None
#Missing values: None
#Usability: Ready to use

# Steps to clean up Diabetes Dataset 
#Remove duplicates from the diabetes dataset: df_diabetes.drop_duplicates(inplace=True)
#Consider class balance checks (e.g., ratio of positive to negative labels)
#Identify categorical features and apply encoding (pd.get_dummies or OrdinalEncoder)
#Explore mode, median, and outliers for inconsistent data (e.g., age = 0)

#1 Remove Duplicates
df_diabetes.drop_duplicates(inplace=True)
#2 Handling any missing values
df_diabetes.fillna(df_diabetes.median(), inplace=True)  # For numeric columns
df_diabetes.fillna("Unknown", inplace=True)    # For categorical columns
#3Check for Inconsistencies
  #Negative ages or values outside expected range
  #Incorrect data types (e.g., numeric coded as string)
#4 Check for Class Imbalance

print(df_diabetes['Diabetes_binary'].value_counts(normalize=True))
print(df_hypertension['Hypertension'].value_counts(normalize=True))
#print(df_ckd['classification'].value_counts(normalize=True)) 

# Encode Categorical Variables
# One-hot encoding (for logistic regression, tree-based models)
df = pd.get_dummies(df_diabetes, drop_first=True)

# Or ordinal encoding if there is a natural order
#print("Arun")
#print(df_ckd.columns)
categorical_cols = df_ckd.select_dtypes(include=['object', 'category']).columns.tolist()
#print("Categorical columns:", categorical_cols)

from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
df_ckd[['DoctorInCharge']] = encoder.fit_transform(df_ckd[['DoctorInCharge']])




===alzheimers-Summary ===
          PatientID          Age       Gender   Ethnicity  \
count   1659.000000  1659.000000  1659.000000  1659.00000   
unique          NaN          NaN          NaN         NaN   
top             NaN          NaN          NaN         NaN   
freq            NaN          NaN          NaN         NaN   
mean     830.000000    54.441230     0.515371     0.71308   
std      479.056364    20.549757     0.499914     1.00043   
min        1.000000    20.000000     0.000000     0.00000   
25%      415.500000    36.000000     0.000000     0.00000   
50%      830.000000    54.000000     1.000000     0.00000   
75%     1244.500000    72.000000     1.000000     1.00000   
max     1659.000000    90.000000     1.000000     3.00000   

        SocioeconomicStatus  EducationLevel          BMI      Smoking  \
count           1659.000000     1659.000000  1659.000000  1659.000000   
unique                  NaN             NaN          NaN          NaN   
top                  

In [3]:
# ===== 1) Load & target
df=df_alzheimers
TARGET = "MMSE"  
assert "MMSE" in df.columns
df = df.dropna(subset=["MMSE"]).copy()
df["Impaired"] = (df["MMSE"] < 24).astype(int)   # 1=impaired
y = df["Impaired"].values
X = df.drop(columns=["Impaired","MMSE"])         # drop MMSE to avoid leakage

# 2) Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Preprocess (trees don’t need scaling, SVMs do)
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols)
    ],
    remainder="drop"
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)




In [4]:
#A) Linear SVM (max-margin linear decision boundary)

# LinearSVC gives margins; wrap with calibration to get probabilities
lin_svm = Pipeline(steps=[
    ("pre", pre),
    ("clf", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000))
])

# Hyperparameter: C (inverse regularization strength)
grid_lin = {"clf__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}
gs_lin = GridSearchCV(lin_svm, grid_lin, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)
gs_lin.fit(X_train, y_train)
print("Linear SVM best params:", gs_lin.best_params_, "CV AUC:", round(gs_lin.best_score_, 4))

# Calibrate post-hoc for better probabilities
cal_lin = CalibratedClassifierCV(gs_lin.best_estimator_, method="sigmoid", cv="prefit")
cal_lin.fit(X_train, y_train)
p_test_lin = cal_lin.predict_proba(X_test)[:,1]

print("Linear SVM  Test ROC-AUC:", roc_auc_score(y_test, p_test_lin))
print("Linear SVM  Test PR-AUC :", average_precision_score(y_test, p_test_lin))
print("Linear SVM  Test Brier  :", brier_score_loss(y_test, p_test_lin))
print("Linear SVM  Report:\n", classification_report(y_test, (p_test_lin>=0.5).astype(int), digits=3))


Linear SVM best params: {'clf__C': 3.0} CV AUC: 0.7588
Linear SVM  Test ROC-AUC: 0.8386167146974064
Linear SVM  Test PR-AUC : 0.9485425103065994
Linear SVM  Test Brier  : 0.114874698906167
Linear SVM  Report:
               precision    recall  f1-score   support

           0      0.826     0.229     0.358        83
           1      0.843     0.988     0.910       347

    accuracy                          0.842       430
   macro avg      0.834     0.609     0.634       430
weighted avg      0.840     0.842     0.803       430



/home/codespace/.local/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [ ]:
#RBF kernel SVM (kernel trick → nonlinear boundary)
# SVC with RBF kernel uses kernel trick: K(x,x') = exp(-gamma * ||x-x'||^2)
rbf_svm = Pipeline(steps=[
    ("pre", pre),
    ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, max_iter=-1))
])

# Tune C (regularization) and gamma (kernel width)
grid_rbf = {
    "clf__C":     [0.1, 1.0, 3.0, 10.0],
    "clf__gamma": ["scale", 0.03, 0.1, 0.3, 1.0]
}
gs_rbf = GridSearchCV(rbf_svm, grid_rbf, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)
gs_rbf.fit(X_train, y_train)
print("RBF SVM best params:", gs_rbf.best_params_, "CV AUC:", round(gs_rbf.best_score_, 4))

p_test_rbf = gs_rbf.best_estimator_.predict_proba(X_test)[:,1]
print("RBF SVM   Test ROC-AUC:", roc_auc_score(y_test, p_test_rbf))
print("RBF SVM   Test PR-AUC :", average_precision_score(y_test, p_test_rbf))
print("RBF SVM   Test Brier  :", brier_score_loss(y_test, p_test_rbf))
print("RBF SVM   Report:\n", classification_report(y_test, (p_test_rbf>=0.5).astype(int), digits=3))



In [ ]:
# compare ROC/PR for the two SVMs on test
fpr_l, tpr_l, _ = roc_curve(y_test, p_test_lin)
fpr_r, tpr_r, _ = roc_curve(y_test, p_test_rbf)
plt.figure(); plt.plot(fpr_l,tpr_l,label="Linear SVM"); plt.plot(fpr_r,tpr_r,label="RBF SVM"); plt.plot([0,1],[0,1])
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC — Test"); plt.legend(); plt.show()

pr_l, rc_l, _ = precision_recall_curve(y_test, p_test_lin)
pr_r, rc_r, _ = precision_recall_curve(y_test, p_test_rbf)
plt.figure(); plt.plot(rc_l,pr_l,label="Linear SVM"); plt.plot(rc_r,pr_r,label="RBF SVM")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR — Test"); plt.legend(); plt.show()
